# PhoBERT ESG Topic Classification

In [1]:
# Configuration
config = {
    "model": {
        "name": "vinai/phobert-large",
        "max_length": 256
    },
    "training": {
        "epochs": 5,
        "batch_size": 16,
        "learning_rate": 2.0e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "use_class_weights": True,
        "weight_method": "sqrt_inverse" # options: inverse, sqrt_inverse, effective
    },
    "paths": {
        # Upload your data/labels folder to Kaggle/Colab, or mount Google Drive
        "train_data": "data/labels/hybrid_train_split.parquet",
        "val_data": "data/labels/hybrid_val_split.parquet",
        "gold_test": "data/labels/topic_gold.parquet",
        "output_dir": "outputs/models/topic_phobert_large"
    }
}

In [2]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import argparse

In [3]:
# Check device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## Load data

In [ ]:
print("Loading hybrid labels...")
df_train = pd.read_parquet(config["paths"]["train_data"])
df_val = pd.read_parquet(config["paths"]["val_data"])

df_train["text"] = df_train.apply(prepare_text, axis=1)
df_train["label"] = df_train["final_topic"].map(LABEL2ID)

df_val["text"] = df_val.apply(prepare_text, axis=1)
df_val["label"] = df_val["final_topic"].map(LABEL2ID)

Loading hybrid labels...


In [ ]:
df_train.head(3)

,sent_id,sentence,ctx_prev,ctx_next,section_title,block_type,doc_id,bank,year,llm_topic,llm_conf,llm_reason,weak_topic,weak_conf,final_topic,label_source,text,label
0,vietcombank_2023_s1686_0,| | 31/12/2023\n Triệu VND | 31/12/2022 \n Tri...,,,Tài sản thuế thu nhập doanh nghiệp hoãn lại,table_like,vietcombank_2023,vietcombank,2023,Non_ESG,1.00,Đây là thông tin về số liệu tài chính thuần tú...,S_labor,0.4,Non_ESG,llm,| | 31/12/2023\n Triệu VND | 31/12/2022 \n Tri...,5
1,bsc_2021_s1516_0,2.26 Ghi nhận doanh thu và thu nhập khác,,,2.26 Ghi nhận doanh thu và thu nhập khác,heading_like,bsc_2021,bsc,2021,Non_ESG,1.00,Đây là nội dung về chính sách kế toán và phươn...,S_labor,0.4,Non_ESG,llm,2.26 Ghi nhận doanh thu và thu nhập khác,5
2,bidv_2021_s1098_0,Tham khảo các thông tin từ nhiều nguồn của các...,,,Xác định các vấn đề trọng yếu,paragraph,bidv_2021,bidv,2021,G,0.95,Câu này mô tả quy trình xác định các vấn đề tr...,S_community,0.4,G,llm,Tham khảo các thông tin từ nhiều nguồn của các...,4


In [ ]:
# Check distribution
print("=== Train Distribution ===")
train_dist = df_train["final_topic"].value_counts()
print(train_dist)

print("\n=== Val Distribution ===")
val_dist = df_val["final_topic"].value_counts()
print(val_dist)

=== Train Distribution ===
final_topic
Non_ESG        1458
G               711
S_product       618
S_community     505
S_labor         431
E               428
Name: count, dtype: int64

=== Val Distribution ===
final_topic
Non_ESG        258
G              125
S_product      109
S_community     89
E               76
S_labor         76
Name: count, dtype: int64


In [ ]:
def prepare_text(row: pd.Series) -> str:
    """Combine sentence with context for better classification"""
    parts = []
    if pd.notna(row.get("ctx_prev")) and row["ctx_prev"]:
        parts.append(str(row["ctx_prev"]))
    parts.append(str(row["sentence"]))
    if pd.notna(row.get("ctx_next")) and row["ctx_next"]:
        parts.append(str(row["ctx_next"]))
    return " ".join(parts)


# Prepare train data
df_train["text"] = df_train.apply(prepare_text, axis=1)
df_train["label"] = df_train["final_topic"].map(LABEL2ID)

# Prepare val data
df_val["text"] = df_val.apply(prepare_text, axis=1)
df_val["label"] = df_val["final_topic"].map(LABEL2ID)

print("Sample text:")
print(df_train["text"].iloc[0][:400])

Sample text:
| | 31/12/2023
 Triệu VND | 31/12/2022 
 Triệu VND |
|----|---------|---------|
| Tài sản thuế thu nhập doanh nghiệp hoãn lại phát sinh từ các khoản 
 chênh lệch tạm thời phải chịu thuế | 848.268 | 958.065 |


In [ ]:
# Check text lengths
train_lens = df_train["text"].str.len()
print(f"Text length stats:")
print(f"  Mean: {train_lens.mean():.0f}")
print(f"  Median: {train_lens.median():.0f}")
print(f"  Max: {train_lens.max():.0f}")
print(f"  95th percentile: {train_lens.quantile(0.95):.0f}")

Text length stats:
  Mean: 416
  Median: 356
  Max: 4181
  95th percentile: 921


## Compute Class Weights

In [ ]:
def compute_class_weights(labels: list, method: str = "inverse") -> torch.Tensor:
    counts = Counter(labels)
    n_samples = len(labels)
    n_classes = len(TOPICS)

    if method == "inverse":
        weights = [n_samples / (n_classes * counts[i]) for i in range(n_classes)]
    elif method == "sqrt_inverse":
        weights = [np.sqrt(n_samples / (n_classes * counts[i])) for i in range(n_classes)]
    elif method == "effective":
        beta = 0.9999
        weights = [(1 - beta) / (1 - beta ** counts[i]) for i in range(n_classes)]
    else:
        weights = [1.0] * n_classes

    weights = torch.tensor(weights, dtype=torch.float32)
    weights = weights / weights.sum() * n_classes
    return weights

In [ ]:
use_weights = config["training"]["use_class_weights"]
weight_method = config["training"]["weight_method"]
class_weights = None
if use_weights:
    class_weights = compute_class_weights(df_train["label"].tolist(), method=weight_method)
    print(f"\nClass weights ({weight_method}):")
    for t, w in zip(TOPICS, class_weights):
        print(f"  {t}: {w:.3f}")


Class weights (sqrt_inverse):
  E: 1.184
  S_labor: 1.180
  S_community: 1.090
  S_product: 0.985
  G: 0.919
  Non_ESG: 0.642


## Dataset & DataLoader

In [ ]:
class TopicDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = 256):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

## Load Model

In [ ]:
TOPICS = ["E", "S_labor", "S_community", "S_product", "G", "Non_ESG"]
LABEL2ID = {t: i for i, t in enumerate(TOPICS)}
ID2LABEL = {i: t for t, i in LABEL2ID.items()}
MODEL_NAME = config["model"]["name"]

In [ ]:
print(f"\nLoading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TOPICS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)


Loading model: vinai/phobert-large


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

In [ ]:
# Create datasets
MAX_LEN = config["model"]["max_length"]
train_dataset = TopicDataset(df_train, tokenizer, MAX_LEN)
val_dataset = TopicDataset(df_val, tokenizer, MAX_LEN)

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Val dataset: {len(val_dataset):,} samples")

# Test one sample
sample = train_dataset[0]
print(f"\nSample shape:")
print(f"  input_ids: {sample['input_ids'].shape}")
print(f"  attention_mask: {sample['attention_mask'].shape}")
print(f"  label: {sample['labels']}")

Train dataset: 4,151 samples
Val dataset: 733 samples

Sample shape:
  input_ids: torch.Size([256])
  attention_mask: torch.Size([256])
  label: 5


## Custom Trainer with Class Weights

In [ ]:
class WeightedTrainer(Trainer):

    def __init__(self, class_weights: torch.Tensor = None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            weight = self.class_weights.to(logits.device)
            loss_fn = nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fn = nn.CrossEntropyLoss()

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    micro_f1 = f1_score(labels, preds, average="micro")
    return {"macro_f1": macro_f1, "micro_f1": micro_f1}

In [ ]:
epochs = config["training"]["epochs"]
batch_size = config["training"]["batch_size"]
lr = config["training"]["learning_rate"]
output_dir = Path(config["paths"]["output_dir"])
weight_decay = config["training"]["weight_decay"]
warmup_ratio = config["training"]["warmup_ratio"]

training_args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 2,
    learning_rate=lr,
    weight_decay=weight_decay,
    warmup_ratio=warmup_ratio,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

print("Training Arguments:")
print(f"  Epochs: {epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {lr}")
print(f"  Warmup ratio: {warmup_ratio}")
print(f"  FP16: {training_args.fp16}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training Arguments:
  Epochs: 5
  Batch size: 16
  Learning rate: 2e-05
  Warmup ratio: 0.1
  FP16: True


In [ ]:
trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("\nTraining...")
trainer.train()


Training...


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1
1,0.658241,0.682520,0.757961,0.761255
2,0.467583,0.575514,0.811467,0.810368
3,0.345210,0.582441,0.833189,0.837653
4,0.213685,0.674349,0.826473,0.821282
5,0.102528,0.682163,0.836867,0.839018


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1300, training_loss=0.48250366357656627, metrics={'train_runtime': 1374.6353, 'train_samples_per_second': 15.099, 'train_steps_per_second': 0.946, 'total_flos': 9671248429624320.0, 'train_loss': 0.48250366357656627, 'epoch': 5.0})

In [ ]:

print("\n=== Validation Results ===")
val_results = trainer.evaluate()
print(val_results)

# Gold test evaluation (if available)
gold_path = Path(config["paths"]["gold_test"])
if gold_path.exists():
    try:
        df_gold = pd.read_parquet(gold_path)
        # If CSV, we could use read_csv, but notebook uses parquet. We'll handle both.
        if gold_path.suffix == '.csv':
              df_gold = pd.read_csv(gold_path)

        # ensure gold_topic exists
        if "gold_topic" in df_gold.columns:
            df_gold = df_gold[df_gold["gold_topic"].notna() & (df_gold["gold_topic"] != "")]
            df_gold["text"] = df_gold.apply(prepare_text, axis=1)
            df_gold["label"] = df_gold["gold_topic"].map(LABEL2ID)

            if len(df_gold) > 0:
                print(f"\n=== Gold Test Results ({len(df_gold)} samples) ===")
                test_dataset = TopicDataset(df_gold, tokenizer, max_len)
                preds = trainer.predict(test_dataset)
                y_pred = np.argmax(preds.predictions, axis=-1)
                y_true = df_gold["label"].values

                print(f"Macro-F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
                print("\nClassification Report:")
                print(classification_report(y_true, y_pred, target_names=TOPICS))
    except Exception as e:
        print(f"Gold test skipped: {e}")

# Save model
final_path = output_dir / "final"
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"\nModel saved: {final_path}")


=== Validation Results ===


{'eval_loss': 0.6821630001068115, 'eval_macro_f1': 0.8368667329037568, 'eval_micro_f1': 0.8390177353342428, 'eval_runtime': 9.389, 'eval_samples_per_second': 78.07, 'eval_steps_per_second': 2.45, 'epoch': 5.0}

=== Gold Test Results (500 samples) ===
Macro-F1: 0.9502

Classification Report:
              precision    recall  f1-score   support

           E       0.94      0.96      0.95        80
     S_labor       0.94      0.95      0.94        80
 S_community       0.96      0.96      0.96        80
   S_product       0.96      0.95      0.96        80
           G       0.97      0.92      0.94        90
     Non_ESG       0.93      0.96      0.95        90

    accuracy                           0.95       500
   macro avg       0.95      0.95      0.95       500
weighted avg       0.95      0.95      0.95       500



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved: outputs/models/topic_phobert_large/final


In [ ]:
def predict_topic(text: str, model, tokenizer, device="cpu") -> dict:
    """Predict topic for a single text"""
    model.eval()
    model.to(device)

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
        conf = probs[0][pred_id].item()

    return {
        "topic": ID2LABEL[pred_id],
        "confidence": conf,
        "all_probs": {ID2LABEL[i]: round(p, 4) for i, p in enumerate(probs[0].tolist())}
    }

In [ ]:
# Test samples
test_texts = [
    "Ngân hàng đã giảm phát thải carbon 20% so với năm trước thông qua sử dụng năng lượng tái tạo.",
    "Tổng số nhân viên tăng 15%, trong đó tỷ lệ nữ lãnh đạo đạt 35%.",
    "Chương trình học bổng đã hỗ trợ 500 sinh viên nghèo trong năm qua.",
    "Ứng dụng mobile banking được nâng cấp với nhiều tính năng bảo mật mới.",
    "Hội đồng quản trị họp định kỳ hàng quý để giám sát hoạt động kinh doanh.",
    "Doanh thu năm 2023 đạt 50.000 tỷ đồng, tăng 12% so với cùng kỳ.",
]

print("=== Test ===")
for text in test_texts:
    result = predict_topic(text, model, tokenizer, device=DEVICE)
    print(f"\nText: {text[:80]}...")
    print(f"  → {result['topic']} ({result['confidence']:.2%})")

=== Test ===

Text: Ngân hàng đã giảm phát thải carbon 20% so với năm trước thông qua sử dụng năng l...
  → E (99.52%)

Text: Tổng số nhân viên tăng 15%, trong đó tỷ lệ nữ lãnh đạo đạt 35%....
  → S_labor (99.55%)

Text: Chương trình học bổng đã hỗ trợ 500 sinh viên nghèo trong năm qua....
  → S_community (99.10%)

Text: Ứng dụng mobile banking được nâng cấp với nhiều tính năng bảo mật mới....
  → S_product (99.36%)

Text: Hội đồng quản trị họp định kỳ hàng quý để giám sát hoạt động kinh doanh....
  → Non_ESG (98.23%)

Text: Doanh thu năm 2023 đạt 50.000 tỷ đồng, tăng 12% so với cùng kỳ....
  → Non_ESG (99.43%)


In [ ]:
# Config for HuggingFace Hub
HF_REPO_NAME = "esg-topic-classifier"
HF_USERNAME = "huypham71"

# Model card info
MODEL_CARD_INFO = {
    "language": ["vi"],
    "license": "mit",
    "tags": ["text-classification", "phobert", "vietnamese", "esg", "sustainability"],
    "datasets": ["custom"],
    "metrics": ["f1"],
}

In [ ]:
from huggingface_hub import login, HfApi, whoami

try:
    user_info = whoami()
    HF_USERNAME = user_info["name"]
    print(f"✅ Already logged in as: {HF_USERNAME}")
except Exception:
    print("🔐 Please login to HuggingFace Hub...")
    print("   You can get your token from: https://huggingface.co/settings/tokens")
    login()
    user_info = whoami()
    HF_USERNAME = user_info["name"]
    print(f"✅ Logged in as: {HF_USERNAME}")

✅ Already logged in as: huypham71


In [ ]:
# Create model card content
model_card_content = f"""
"""
# Save model card
model_card_path = final_path / "README.md"
with open(model_card_path, "w", encoding="utf-8") as f:
    f.write(model_card_content)

In [ ]:
# Push model to HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi()
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f"Pushing model to: https://huggingface.co/{repo_id}")
print(f"Source: outputs/models/topic_phobert_large/hybrid/final")

# Create repo if not exists and upload
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
    private=False,  # Set to True if you want private repo
)

# Upload all files in OUT_DIR
api.upload_folder(
    folder_path=final_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message=f"Upload ESG Topic Classifier",
)

Pushing model to: https://huggingface.co/huypham71/esg-topic-classifier
Source: outputs/models/topic_phobert_large/hybrid/final


/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:10176: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e/final/model.safetensors:   0%|          |  558kB / 1.48GB            

  ...e/final/training_args.bin:  73%|#######3  | 3.81kB / 5.20kB            

CommitInfo(commit_url='https://huggingface.co/huypham71/esg-topic-classifier/commit/4a968536d670b755f21f3fc6ffb259de13202ed5', commit_message='Upload ESG Topic Classifier', commit_description='', oid='4a968536d670b755f21f3fc6ffb259de13202ed5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/huypham71/esg-topic-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='huypham71/esg-topic-classifier'), pr_revision=None, pr_num=None)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "huypham71/esg-topic-classifier"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "Ngân hàng cam kết giảm 20% lượng khí thải carbon vào năm 2025."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)

with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]

topk = torch.topk(probs, k=3)
for score, idx in zip(topk.values, topk.indices):
    print(model.config.id2label[idx.item()], float(score))


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

E 0.9951243996620178
S_labor 0.0014044769341126084
G 0.0011063175043091178


## Inference

In [10]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import argparse

SENTENCES_PATH = Path("data/corpus/sentences.parquet")
WEAK_LABELS_PATH = Path("data/labels/weak_labels.parquet")
OUTPUT_PATH = Path("data/corpus/sentences_with_topic.parquet")

TOPICS = ["E", "S_labor", "S_community", "S_product", "G", "Non_ESG"]
LABEL2ID = {t: i for i, t in enumerate(TOPICS)}
ID2LABEL = {i: t for t, i in LABEL2ID.items()}

NON_ESG_ID = LABEL2ID["Non_ESG"]


def prepare_text(row: pd.Series) -> str:
    parts = []
    if pd.notna(row.get("ctx_prev")) and row["ctx_prev"]:
        parts.append(str(row["ctx_prev"]))
    parts.append(str(row["sentence"]))
    if pd.notna(row.get("ctx_next")) and row["ctx_next"]:
        parts.append(str(row["ctx_next"]))
    return " ".join(parts)


def load_model(device: str = "cuda"):
    model_name = "huypham71/esg-topic-classifier"
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return tokenizer, model


def batch_inference(
    texts: list,
    tokenizer,
    model,
    batch_size: int = 32,
    max_len: int = 256,
    device: str = "cuda",
) -> tuple[list, list]:
    all_preds = []
    all_probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Inference"):
        batch_texts = texts[i:i + batch_size]

        encoding = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt",
        )
        encoding = {k: v.to(device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = model(**encoding)
            probs = torch.softmax(outputs.logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    return all_preds, all_probs


def apply_hard_overrides(
    df: pd.DataFrame,
    preds: list,
    probs: list,
    weak_df: pd.DataFrame = None,
) -> pd.DataFrame:
    df["topic_pred_raw"] = [ID2LABEL[p] for p in preds]
    df["topic_prob_raw"] = [max(p) for p in probs]
    df["topic_probs"] = probs

    # Initialize final predictions
    df["topic_pred"] = df["topic_pred_raw"]
    df["topic_prob"] = df["topic_prob_raw"]
    df["override_reason"] = ""

    # meta_heading -> Non_ESG
    mask_meta = df["block_type"] == "meta_heading"
    df.loc[mask_meta, "topic_pred"] = "Non_ESG"
    df.loc[mask_meta, "topic_prob"] = 1.0
    df.loc[mask_meta, "override_reason"] = "meta_heading"

    # Merge weak labels for trivial Non_ESG detection
    if weak_df is not None:
        weak_subset = weak_df[["sent_id", "weak_topic", "weak_conf"]].copy()
        df = df.merge(weak_subset, on="sent_id", how="left")

        # Trivial Non_ESG: weak says Non_ESG with high conf AND model uncertain
        mask_trivial = (
            (df["weak_topic"] == "Non_ESG") &
            (df["weak_conf"] >= 0.8) &
            (df["topic_prob_raw"] < 0.6) &
            (df["override_reason"] == "")
        )
        df.loc[mask_trivial, "topic_pred"] = "Non_ESG"
        df.loc[mask_trivial, "topic_prob"] = df.loc[mask_trivial, "weak_conf"]
        df.loc[mask_trivial, "override_reason"] = "weak_trivial_non_esg"

    # Clean up intermediate columns
    df["topic_probs"] = df["topic_probs"].apply(
        lambda p: {ID2LABEL[i]: round(v, 4) for i, v in enumerate(p)}
    )

    return df


def run_inference(
    batch_size: int = 32,
    max_len: int = 256,
    device: str = None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

    print(f"Device: {device}")

    # Load data
    print("Loading sentences...")
    df = pd.read_parquet(SENTENCES_PATH)
    print(f"Total sentences: {len(df)}")

    # Load weak labels for override
    weak_df = None
    if WEAK_LABELS_PATH.exists():
        weak_df = pd.read_parquet(WEAK_LABELS_PATH)
        print(f"Loaded weak labels for override: {len(weak_df)}")

    # Prepare texts
    print("Preparing texts...")
    df["text"] = df.apply(prepare_text, axis=1)
    texts = df["text"].tolist()

    # Load model and run inference
    tokenizer, model = load_model(device)
    preds, probs = batch_inference(texts, tokenizer, model, batch_size, max_len, device)

    # Apply hard overrides
    print("Applying hard overrides...")
    df = apply_hard_overrides(df, preds, probs, weak_df)

    # Remove temporary columns
    cols_to_drop = ["text", "topic_pred_raw", "topic_prob_raw"]
    if weak_df is not None:
        cols_to_drop.extend(["weak_topic", "weak_conf"])
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # Save results
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(OUTPUT_PATH, index=False)

    print(f"\n=== INFERENCE COMPLETE ===")
    print(f"Output: {OUTPUT_PATH}")
    print(f"\nTopic Distribution:")
    print(df["topic_pred"].value_counts())

    print(f"\nOverride Summary:")
    print(df["override_reason"].value_counts())

    # Statistics
    print(f"\nConfidence Statistics:")
    print(df.groupby("topic_pred")["topic_prob"].describe().round(3))

    return df

In [11]:
run_inference()

Device: cuda
Loading sentences...
Total sentences: 95133
Loaded weak labels for override: 94943
Preparing texts...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Inference: 100%|██████████| 2973/2973 [59:10<00:00,  1.19s/it]


Applying hard overrides...

=== INFERENCE COMPLETE ===
Output: data/corpus/sentences_with_topic.parquet

Topic Distribution:
topic_pred
Non_ESG        50209
G              21527
S_product       8881
S_labor         6883
E               4699
S_community     2934
Name: count, dtype: int64

Override Summary:
override_reason
    95133
Name: count, dtype: int64

Confidence Statistics:
               count   mean    std    min    25%    50%    75%    max
topic_pred                                                           
E             4699.0  0.952  0.108  0.320  0.981  0.994  0.995  0.996
G            21527.0  0.942  0.119  0.203  0.968  0.993  0.995  0.996
Non_ESG      50209.0  0.970  0.081  0.226  0.992  0.995  0.995  0.996
S_community   2934.0  0.905  0.187  0.193  0.945  0.992  0.994  0.995
S_labor       6883.0  0.959  0.104  0.197  0.990  0.995  0.996  0.996
S_product     8881.0  0.945  0.115  0.229  0.972  0.993  0.995  0.995


,doc_id,bank,year,section_id,block_id,sent_id,sent_idx_in_block,sentence,ctx_prev,ctx_next,block_type,section_title,topic_probs,topic_pred,topic_prob,override_reason
0,agribank_2020,agribank,2020,agribank_2020_sec6,agribank_2020_b28,agribank_2020_s28_0,0,| THÔNG ĐIỆP CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊ\n...,,,table_like,MỤC LỤC,"{'E': 0.0029, 'S_labor': 0.0038, 'S_community'...",Non_ESG,0.922011,
1,agribank_2020,agribank,2020,agribank_2020_sec6,agribank_2020_b29,agribank_2020_s29_0,0,| ĐÁNH GIÁ CỦA HỘI ĐỒNG THÀNH VIÊN | 48 |\n|--...,,,table_like,MỤC LỤC,"{'E': 0.0013, 'S_labor': 0.0007, 'S_community'...",Non_ESG,0.995143,
2,agribank_2020,agribank,2020,agribank_2020_sec8,agribank_2020_b34,agribank_2020_s34_0,0,"Năm 2020, chúng ta đã chứng kiến Việt Nam tạo ...",,"Ngành Ngân hàng, với vai trò huyết mạch của nề...",paragraph,CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊN,"{'E': 0.0015, 'S_labor': 0.0007, 'S_community'...",Non_ESG,0.994283,
3,agribank_2020,agribank,2020,agribank_2020_sec8,agribank_2020_b34,agribank_2020_s34_1,1,"Ngành Ngân hàng, với vai trò huyết mạch của nề...","Năm 2020, chúng ta đã chứng kiến Việt Nam tạo ...",,paragraph,CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊN,"{'E': 0.0015, 'S_labor': 0.0007, 'S_community'...",Non_ESG,0.994283,
4,agribank_2020,agribank,2020,agribank_2020_sec8,agribank_2020_b35,agribank_2020_s35_0,0,"Cùng đất nước và ngành ngân hàng, Agribank tự ...",,"Đến 31/12/2020, tổng tài sản của Agribank đạt ...",kpi_like,CỦA CHỦ TỊCH HỘI ĐỒNG THÀNH VIÊN,"{'E': 0.0067, 'S_labor': 0.0018, 'S_community'...",Non_ESG,0.717986,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95128,vpbank_2024,vpbank,2024,vpbank_2024_sec621,vpbank_2024_b2985,vpbank_2024_s2985_2,2,GPBank là pháp nhân độc lập và không thực hiện...,"Sau chuyển giao bắt buộc, GPBank tiếp tục hoạt...",,kpi_like,49. Các sự kiện sau ngày kết thúc năm tài chính,"{'E': 0.0011, 'S_labor': 0.0006, 'S_community'...",Non_ESG,0.990644,
95129,vpbank_2024,vpbank,2024,vpbank_2024_sec623,vpbank_2024_b2988,vpbank_2024_s2988_0,0,Tỷ giá một số loại ngoại tệ và vàng so với VND...,,,heading_like,50. Tỷ giá một số loại ngoại tệ và vàng so với...,"{'E': 0.0015, 'S_labor': 0.001, 'S_community':...",Non_ESG,0.994893,
95130,vpbank_2024,vpbank,2024,vpbank_2024_sec623,vpbank_2024_b2990,vpbank_2024_s2990_0,0,THUYẾT MINH BÁO CÁO TÀI CHÍNH HỢP NHẤT (tiếp t...,,,paragraph,50. Tỷ giá một số loại ngoại tệ và vàng so với...,"{'E': 0.0011, 'S_labor': 0.0007, 'S_community'...",Non_ESG,0.995657,
95131,vpbank_2024,vpbank,2024,vpbank_2024_sec623,vpbank_2024_b2991,vpbank_2024_s2991_0,0,Không có sự kiện trọng yếu nào xảy ra kể từ ng...,,,paragraph,50. Tỷ giá một số loại ngoại tệ và vàng so với...,"{'E': 0.0009, 'S_labor': 0.0006, 'S_community'...",Non_ESG,0.990778,
